# H$_2$ VQE on Neutral-Atom Hardware

**Chemistry → Mapping → Noise → Validation** 📐

This notebook presents a minimal, application-facing workflow for mapping a small quantum chemistry problem to a neutral-atom-style execution pipeline. The emphasis is not only ideal-state optimization, but also identifying **where execution remains stable under noise and hardware-aware constraints**.


## 1. Imports

We use only lightweight scientific Python packages so the notebook stays easy to run and easy to understand.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


## 2. Minimal H$_2$ Hamiltonian

For this first notebook, we use a compact 2-qubit toy Hamiltonian with coefficients often used in introductory H$_2$ demonstrations. This keeps the workflow focused on the full pipeline:

- chemistry target
- variational state
- neutral-atom-style mapping proxy
- noise sweep
- constraint-based validation


In [ ]:
H2_TERMS = [
    ("II", -1.0523732458),
    ("ZI",  0.3979374248),
    ("IZ", -0.3979374248),
    ("ZZ", -0.0112801043),
    ("XX",  0.1809311998),
]

H2_TERMS


## 3. Pauli operators and expectation values

We define a small amount of linear algebra to evaluate energies exactly for 2-qubit states.

In [ ]:
I = np.array([[1, 0], [0, 1]], dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

PAULI = {
    "I": I,
    "X": X,
    "Y": Y,
    "Z": Z,
}


def kron2(a, b):
    return np.kron(a, b)


def pauli_string_matrix(label: str) -> np.ndarray:
    if len(label) != 2:
        raise ValueError("This notebook expects 2-qubit Pauli strings.")
    return kron2(PAULI[label[0]], PAULI[label[1]])


def hamiltonian_matrix(terms) -> np.ndarray:
    H = np.zeros((4, 4), dtype=complex)
    for label, coeff in terms:
        H = H + coeff * pauli_string_matrix(label)
    return H


def expectation_value(state: np.ndarray, operator: np.ndarray) -> float:
    value = np.vdot(state, operator @ state)
    return float(np.real_if_close(value))


H = hamiltonian_matrix(H2_TERMS)
H


## 4. Variational ansatz

We use a deliberately small real-valued ansatz:

\[
|\psi(\theta)\rangle = \cos(\theta/2)|00\rangle + \sin(\theta/2)|11\rangle.
\]

This is enough to support a clean first end-to-end workflow.

In [ ]:
def ansatz_state(theta: float) -> np.ndarray:
    state = np.array([
        np.cos(theta / 2.0),
        0.0,
        0.0,
        np.sin(theta / 2.0),
    ], dtype=complex)
    norm = np.linalg.norm(state)
    if norm == 0:
        raise ValueError("State norm is zero.")
    return state / norm


theta_demo = 1.0
psi_demo = ansatz_state(theta_demo)
psi_demo


## 5. Neutral-atom-style mapping proxy

This first notebook keeps the mapping layer simple. We define a small layout object with:

- number of atoms
- spacing
- blockade radius

This is not a full pulse-level compiler; it is a lightweight hardware-facing check that keeps the repo aligned with neutral-atom application workflows.

In [ ]:
class NeutralAtomLayout:
    def __init__(self, n_atoms: int, spacing_um: float, blockade_radius_um: float):
        self.n_atoms = n_atoms
        self.spacing_um = spacing_um
        self.blockade_radius_um = blockade_radius_um

    def blockade_active(self) -> bool:
        return self.spacing_um <= self.blockade_radius_um

    def summary(self) -> dict:
        return {
            "n_atoms": self.n_atoms,
            "spacing_um": self.spacing_um,
            "blockade_radius_um": self.blockade_radius_um,
            "blockade_active": self.blockade_active(),
        }


layout = NeutralAtomLayout(n_atoms=2, spacing_um=6.0, blockade_radius_um=7.0)
layout.summary()


## 6. Noise model

For a minimal first notebook, we use a simple **coherence-loss proxy** instead of a full Lindblad solver. The idea is to interpolate from the variational state toward the reference state \(|00\rangle\) as noise increases.

This preserves a clear story:

- ideal chemistry state at low noise
- degraded execution at higher noise
- validation threshold that identifies stable operating regions


In [ ]:
REFERENCE_STATE = np.array([1.0, 0.0, 0.0, 0.0], dtype=complex)


def apply_noise_proxy(state: np.ndarray, gamma: float) -> np.ndarray:
    """Blend the ansatz state toward |00> as gamma increases.

    gamma = 0   -> original state
    gamma large -> increasingly dominated by |00>
    """
    weight_signal = np.exp(-gamma)
    weight_reference = 1.0 - np.exp(-gamma)
    noisy = weight_signal * state + weight_reference * REFERENCE_STATE
    norm = np.linalg.norm(noisy)
    if norm == 0:
        raise ValueError("Noisy state norm is zero.")
    return noisy / norm


for g in [0.0, 0.25, 0.5, 1.0]:
    print(g, apply_noise_proxy(psi_demo, g))


## 7. Constraint-based validation 📐

We now define the validation layer. The notebook uses an overlap-based stability score against the reference state:

\[
\cos\theta = \frac{\langle r | \psi \rangle}{\|r\|\,\|\psi\|}
\]

with threshold

\[
\cos\theta \geq \frac{1}{\sqrt{1^2 + 1^2}}.
\]

This gives a compact **constraint gate** for deciding whether a state remains within a chosen stable execution region.

In [ ]:
THRESHOLD = 1.0 / np.sqrt(1.0**2 + 1.0**2)


def constraint_score(state: np.ndarray, reference: np.ndarray = REFERENCE_STATE) -> float:
    denom = np.linalg.norm(state) * np.linalg.norm(reference)
    if denom == 0:
        raise ValueError("Zero norm encountered in constraint_score.")
    value = np.vdot(reference, state) / denom
    return float(np.real_if_close(value))


def passes_constraint(state: np.ndarray, threshold: float = THRESHOLD) -> bool:
    return constraint_score(state) >= threshold


print("Threshold:", THRESHOLD)
print("Constraint score (ideal state):", constraint_score(psi_demo))
print("Passes?", passes_constraint(psi_demo))


## 8. Energy evaluation and parameter sweep

We now scan both:

- variational angle \(\theta\)
- noise level \(\gamma\)

For each pair, we compute:

- noisy state
- energy
- constraint score
- pass/fail decision


In [ ]:
def energy_of_state(state: np.ndarray) -> float:
    return expectation_value(state, H)


def evaluate_theta_gamma(theta: float, gamma: float) -> dict:
    state = ansatz_state(theta)
    noisy_state = apply_noise_proxy(state, gamma)
    return {
        "theta": float(theta),
        "gamma": float(gamma),
        "energy": energy_of_state(noisy_state),
        "constraint_score": constraint_score(noisy_state),
        "passes": passes_constraint(noisy_state),
        "state": noisy_state,
    }


theta_grid = np.linspace(0.0, np.pi, 200)
gamma_grid = np.linspace(0.0, 1.25, 60)

energy_surface = np.zeros((len(gamma_grid), len(theta_grid)))
score_surface = np.zeros((len(gamma_grid), len(theta_grid)))
pass_surface = np.zeros((len(gamma_grid), len(theta_grid)), dtype=bool)

for i, gamma in enumerate(gamma_grid):
    for j, theta in enumerate(theta_grid):
        result = evaluate_theta_gamma(theta, gamma)
        energy_surface[i, j] = result["energy"]
        score_surface[i, j] = result["constraint_score"]
        pass_surface[i, j] = result["passes"]


## 9. Best energy vs noise

At each noise level, we minimize over the variational parameter and compare:

- **best overall energy**
- **best energy among constraint-passing states**

This makes the validation layer visible at the application level.

In [ ]:
best_energy_all = []
best_theta_all = []

best_energy_valid = []
best_theta_valid = []
valid_exists = []

for i, gamma in enumerate(gamma_grid):
    energies = energy_surface[i]
    scores = score_surface[i]
    passes = pass_surface[i]

    idx_all = int(np.argmin(energies))
    best_energy_all.append(energies[idx_all])
    best_theta_all.append(theta_grid[idx_all])

    if np.any(passes):
        valid_energies = np.where(passes, energies, np.inf)
        idx_valid = int(np.argmin(valid_energies))
        best_energy_valid.append(energies[idx_valid])
        best_theta_valid.append(theta_grid[idx_valid])
        valid_exists.append(True)
    else:
        best_energy_valid.append(np.nan)
        best_theta_valid.append(np.nan)
        valid_exists.append(False)

best_energy_all = np.array(best_energy_all)
best_theta_all = np.array(best_theta_all)
best_energy_valid = np.array(best_energy_valid)
best_theta_valid = np.array(best_theta_valid)
valid_exists = np.array(valid_exists)


## 10. Plot: best energy under noise

This plot gives the main application-facing summary for Notebook 01.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(gamma_grid, best_energy_all, label="Best energy (all states)")
plt.plot(gamma_grid, best_energy_valid, label="Best energy (constraint-passing)")
plt.axhline(np.min(np.linalg.eigvalsh(H)), linestyle="--", label="Ground-state energy of toy Hamiltonian")
plt.xlabel("Noise γ")
plt.ylabel("Energy")
plt.title("H₂ VQE under noise with constraint-based validation")
plt.legend()
plt.tight_layout()
plt.show()


## 11. Plot: stable region in $(\gamma, \theta)$ space

This figure shows which parameter regions remain inside the chosen validation threshold.

In [ ]:
plt.figure(figsize=(8, 5))
plt.imshow(
    pass_surface.astype(float),
    aspect="auto",
    origin="lower",
    extent=[theta_grid.min(), theta_grid.max(), gamma_grid.min(), gamma_grid.max()],
)
plt.colorbar(label="Constraint pass (1=yes, 0=no)")
plt.xlabel("θ")
plt.ylabel("Noise γ")
plt.title("Stable execution region under overlap-based constraint gate")
plt.tight_layout()
plt.show()


## 12. Plot: best variational parameter vs noise

This plot shows how the optimal variational angle shifts as noise changes.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(gamma_grid, best_theta_all, label="Best θ (all states)")
plt.plot(gamma_grid, best_theta_valid, label="Best θ (constraint-passing)")
plt.xlabel("Noise γ")
plt.ylabel("Optimal θ")
plt.title("Best variational parameter under noise")
plt.legend()
plt.tight_layout()
plt.show()


## 13. Save a figure for the repo

This cell saves a clean figure to `../figures/` when run from the `notebooks/` directory inside the repo.

In [ ]:
import os

output_path = os.path.join("..", "figures", "h2_energy_vs_noise.png")

plt.figure(figsize=(8, 5))
plt.plot(gamma_grid, best_energy_all, label="Best energy (all states)")
plt.plot(gamma_grid, best_energy_valid, label="Best energy (constraint-passing)")
plt.axhline(np.min(np.linalg.eigvalsh(H)), linestyle="--", label="Ground-state energy of toy Hamiltonian")
plt.xlabel("Noise γ")
plt.ylabel("Energy")
plt.title("H₂ VQE under noise with constraint-based validation")
plt.legend()
plt.tight_layout()
plt.savefig(output_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved figure to: {output_path}")


## 14. Summary

This first notebook establishes a minimal application-facing pipeline:

- define a chemistry target
- construct a variational state
- include a neutral-atom-style mapping proxy
- sweep noise
- validate execution with a constraint gate

In later notebooks, this can be extended toward:

- richer geometry studies
- blockade-aware layout comparisons
- stronger noise models
- larger chemistry examples
- fault-tolerance-oriented workflows


## Notes

This workflow can also be read as a three-stage pipeline:

- algorithm definition
- hardware mapping
- execution validation

That structure generalizes beyond this notebook to broader quantum applications work.